# Shortest routes between all point pairs on a line network

This script finds the shortest route along a vector network between pairs of points and extracts the line segments those routes use. Points are snapped to the network if they are not in exact allignment.

**How to use it:**
1. Edit the settings cell below (file paths etc.).
2. Run all cells (*Run → Run All Cells*).

Requirements: `pip install geopandas networkx "shapely>=2.0" matplotlib`

---
## 1. Settings

In [ ]:
# ====================================================================
#  INPUT FILES
# ====================================================================
NETWORK_FILE = r"Itiner_e_Select.geojson"
POINTS_FILE  = r"Node_sites.shp"

# ====================================================================
#  OUTPUT FOLDER
# ====================================================================
OUT_DIR = r"output"

# ====================================================================
#  OPTIONS
# ====================================================================
ID_FIELD = None                # Attribute field in points file to use as identifier (None defaults to row number)
MAX_SNAP = None                # threshold for snapping points. Uses map units. If a point is further than this from the netwrok it is not snapped (None defaults to alwasy snap)
DECIMALS = 3                   # Coordinate rounding used to join vertices
NODE_INTERSECTIONS = True      # True: crossed lines become junctions False: only shared vertices connect
WORK_CRS = None                # projected CRS for distances, e.g. "EPSG:25831". None = auto UTM if data is lat/lon

---
## 2. Imports and functions

In [ ]:
import math
import os
import warnings
from collections import defaultdict

import geopandas as gpd
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
from shapely.geometry import LineString
from shapely.ops import linemerge, unary_union
from shapely.strtree import STRtree

In [ ]:
def make_key(decimals):
    """Round coordinates so vertices that should coincide become the same graph node."""
    def key(xy):
        return (round(xy[0], decimals), round(xy[1], decimals))
    return key


def iter_lines(geom):
    """Yield LineStrings from any (multi)line or collection geometry."""
    if geom is None or geom.is_empty:
        return
    if geom.geom_type == "LineString":
        yield geom
    elif hasattr(geom, "geoms"):
        for g in geom.geoms:
            yield from iter_lines(g)


def build_graph(lines_gdf, key, node_intersections=True):
    """Turn the line network into a weighted graph (weight = segment length)."""
    geoms = [g for g in lines_gdf.geometry if g is not None and not g.is_empty]
    if node_intersections:
        parts = list(iter_lines(unary_union(geoms)))   # split lines at every crossing
    else:
        parts = [ln for g in geoms for ln in iter_lines(g)]

    G = nx.Graph()
    for ln in parts:
        coords = list(ln.coords)
        for a, b in zip(coords[:-1], coords[1:]):
            ka, kb = key(a), key(b)
            if ka == kb:
                continue
            w = math.dist(ka, kb)
            if G.has_edge(ka, kb) and G[ka][kb]["weight"] <= w:
                continue
            G.add_edge(ka, kb, weight=w)
    return G


def snap_points(G, points_gdf, ids, key, max_snap=None):
    """Snap each point to the nearest spot on the network, splitting the segment there."""
    edges = list(G.edges())
    seg_geoms = [LineString([u, v]) for u, v in edges]
    tree = STRtree(seg_geoms)
    nearest_idx = tree.nearest(np.asarray(points_gdf.geometry.values))

    on_edge = defaultdict(list)
    for pid, geom, ei in zip(ids, points_gdf.geometry, nearest_idx):
        seg = seg_geoms[ei]
        d = seg.distance(geom)
        if max_snap is not None and d > max_snap:
            warnings.warn(f"Point {pid!r} is {d:.1f} units from the network "
                          f"(> MAX_SNAP {max_snap}); skipped.")
            continue
        on_edge[ei].append((seg.project(geom), pid))

    point_node = {}
    for ei, items in on_edge.items():
        u, v = edges[ei]
        seg = seg_geoms[ei]
        items.sort()
        chain = [u]
        for t, pid in items:
            p = seg.interpolate(t)
            node = key((p.x, p.y))
            point_node[pid] = node
            if node != chain[-1] and node != v:
                chain.append(node)
        chain.append(v)
        if len(chain) > 2:
            G.remove_edge(u, v)
            for a, b in zip(chain[:-1], chain[1:]):
                if a != b:
                    G.add_edge(a, b, weight=math.dist(a, b))
    return point_node


def all_pair_routes(G, point_node):
    """Shortest path between every unique pair of points (one Dijkstra run per point).
    """
    import time
    pids = list(point_node)
    print(f"Computing {len(pids) * (len(pids) - 1) // 2} routes between {len(pids)} points...")

    routes, edge_use, unreachable = [], defaultdict(int), []
    t0 = time.time()
    for i, a in enumerate(pids):
        targets = pids[i + 1:]
        if not targets:
            break
        src = point_node[a]
        pred, dist = nx.dijkstra_predecessor_and_distance(G, src, weight="weight")
        for b in targets:
            dst = point_node[b]
            if dst not in dist:
                unreachable.append((a, b))
                continue
            if dst == src:
                continue
            path = [dst]
            while path[-1] != src:
                path.append(pred[path[-1]][0])
            path.reverse()
            routes.append({"from_id": a, "to_id": b, "length_m": dist[dst],
                           "geometry": LineString(path)})
            for u, v in zip(path[:-1], path[1:]):
                edge_use[tuple(sorted((u, v)))] += 1
        elapsed = time.time() - t0
        remaining = elapsed / (i + 1) * (len(pids) - 2 - i)
        print(f"  {i + 1}/{len(pids) - 1} sources done | elapsed {elapsed:.0f}s | "
              f"~{max(remaining, 0):.0f}s remaining   ", end="\r")
    print()
    if unreachable:
        print(f"WARNING: {len(unreachable)} pairs are not connected on the network, e.g. "
              f"{unreachable[:5]}. Check for gaps, or try a lower DECIMALS.")
    return routes, edge_use


---
## 3. Load the data

In [ ]:
lines = gpd.read_file(NETWORK_FILE)
points = gpd.read_file(POINTS_FILE)
lines = lines[lines.geometry.notna() & ~lines.geometry.is_empty].copy()
points = points[points.geometry.notna() & ~points.geometry.is_empty].copy()
out_crs = lines.crs

print(f"Network: {len(lines)} features, CRS = {lines.crs}")
print(f"Points:  {len(points)} features, CRS = {points.crs}")

# Make sure both layers use the same CRS
if lines.crs is None or points.crs is None:
    warnings.warn("One of the inputs has no CRS; assuming both use the same one.")
elif points.crs != lines.crs:
    points = points.to_crs(lines.crs)

# Distances need a CRS in metres
if WORK_CRS:
    work_crs = WORK_CRS
elif lines.crs is not None and lines.crs.is_geographic:
    work_crs = lines.estimate_utm_crs()
    print(f"Data is in lat/lon; working in {work_crs} for metric distances.")
else:
    work_crs = lines.crs
if work_crs is not None and work_crs != lines.crs:
    lines = lines.to_crs(work_crs)
    points = points.to_crs(work_crs)

# Point IDs
if ID_FIELD:
    if ID_FIELD not in points.columns:
        raise ValueError(f"ID_FIELD {ID_FIELD!r} not found. Available fields: {list(points.columns)}")
    ids = points[ID_FIELD].tolist()
    if len(set(ids)) != len(ids):
        raise ValueError(f"ID_FIELD {ID_FIELD!r} contains duplicate values.")
else:
    ids = points.index.tolist()

ax = lines.plot(color="lightgrey", linewidth=0.8, figsize=(8, 8))
points.plot(ax=ax, color="red", markersize=20)
ax.set_title("Input network and points")
plt.show()

---
## 4. Build the network graph and snap the points

In [ ]:
key = make_key(DECIMALS)

G = build_graph(lines, key, node_intersections=NODE_INTERSECTIONS)
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges, "
      f"{nx.number_connected_components(G)} connected component(s)")

point_node = snap_points(G, points, ids, key, MAX_SNAP)
print(f"Snapped {len(point_node)} of {len(points)} points to the network")

comp_of = {}
for ci, comp in enumerate(sorted(nx.connected_components(G), key=len, reverse=True)):
    for n in comp:
        comp_of[n] = ci
pt_comp = defaultdict(list)
for pid, node in point_node.items():
    pt_comp[comp_of[node]].append(pid)
for ci, pids_ in sorted(pt_comp.items()):
    print(f"  Component {ci}: {len(pids_)} point(s) {pids_ if len(pids_) <= 10 else ''}")


---
## 5. Compute the shortest routes

In [ ]:
routes, edge_use = all_pair_routes(G, point_node)
if not routes:
    raise RuntimeError("No routes found.")

routes_gdf = gpd.GeoDataFrame(routes, crs=work_crs)
routes_gdf.head()

---
## 6. Extract the used segments and save

Three output files:
- **routes.geojson**: one line per point pair (`from_id`, `to_id`, `length_m`)
- **used_network.geojson**: unique network stretches used by any route, with `route_count`
- **used_original_lines.geojson**: the original network features touched by a route, with all attributes kept

In [ ]:
by_count = defaultdict(list)
for (u, v), c in edge_use.items():
    by_count[c].append(LineString([u, v]))
used = [{"route_count": c, "length_m": ln.length, "geometry": ln}
        for c, segs in by_count.items() for ln in iter_lines(linemerge(segs))]
used_gdf = gpd.GeoDataFrame(used, crs=work_crs)

tol = 10 ** (-DECIMALS) * 10
orig_tree = STRtree(np.asarray(lines.geometry.values))
used_orig = set()
for (u, v) in edge_use:
    mid = LineString([u, v]).interpolate(0.5, normalized=True)
    hit = orig_tree.query(mid.buffer(tol), predicate="intersects")
    used_orig.update(lines.index[hit])
orig_gdf = lines.loc[sorted(used_orig)]

os.makedirs(OUT_DIR, exist_ok=True)
for gdf, name in [(routes_gdf, "routes"), (used_gdf, "used_network"),
                  (orig_gdf, "used_original_lines")]:
    path = os.path.join(OUT_DIR, f"{name}.geojson")
    (gdf.to_crs(out_crs) if out_crs is not None else gdf).to_file(path, driver="GeoJSON")
    print(f"Saved {len(gdf):>6} features -> {os.path.abspath(path)}")